In [1]:
import os
os.makedirs('charts', exist_ok=True)
print("Charts folder created ✅")

Charts folder created ✅


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
import os
os.makedirs('charts', exist_ok=True)

print("Libraries loaded ✅")

Libraries loaded ✅


In [3]:
df = pd.read_csv(r'C:\Users\USER\Desktop\healthcare-noshow-analysis\KaggleV2-May-2016.csv')
print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
df.head()

Shape: (110527, 14)

Columns: ['PatientId', 'AppointmentID', 'Gender', 'ScheduledDay', 'AppointmentDay', 'Age', 'Neighbourhood', 'Scholarship', 'Hipertension', 'Diabetes', 'Alcoholism', 'Handcap', 'SMS_received', 'No-show']


,PatientId,AppointmentID,Gender,ScheduledDay,AppointmentDay,Age,Neighbourhood,Scholarship,Hipertension,Diabetes,Alcoholism,Handcap,SMS_received,No-show
0,2.987250e+13,5642903,F,2016-04-29T18:38:08Z,2016-04-29T00:00:00Z,62,JARDIM DA PENHA,0,1,0,0,0,0,No
1,5.589978e+14,5642503,M,2016-04-29T16:08:27Z,2016-04-29T00:00:00Z,56,JARDIM DA PENHA,0,0,0,0,0,0,No
2,4.262962e+12,5642549,F,2016-04-29T16:19:04Z,2016-04-29T00:00:00Z,62,MATA DA PRAIA,0,0,0,0,0,0,No
3,8.679512e+11,5642828,F,2016-04-29T17:29:31Z,2016-04-29T00:00:00Z,8,PONTAL DE CAMBURI,0,0,0,0,0,0,No
4,8.841186e+12,5642494,F,2016-04-29T16:07:23Z,2016-04-29T00:00:00Z,56,JARDIM DA PENHA,0,1,1,0,0,0,No


Basic Info

In [4]:
print("Dataset Info:")
print(df.info())
print("\nMissing Values:")
print(df.isnull().sum())
print("\nDuplicates:", df.duplicated().sum())

Dataset Info:
<class 'pandas.DataFrame'>
RangeIndex: 110527 entries, 0 to 110526
Data columns (total 14 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   PatientId       110527 non-null  float64
 1   AppointmentID   110527 non-null  int64  
 2   Gender          110527 non-null  str    
 3   ScheduledDay    110527 non-null  str    
 4   AppointmentDay  110527 non-null  str    
 5   Age             110527 non-null  int64  
 6   Neighbourhood   110527 non-null  str    
 7   Scholarship     110527 non-null  int64  
 8   Hipertension    110527 non-null  int64  
 9   Diabetes        110527 non-null  int64  
 10  Alcoholism      110527 non-null  int64  
 11  Handcap         110527 non-null  int64  
 12  SMS_received    110527 non-null  int64  
 13  No-show         110527 non-null  str    
dtypes: float64(1), int64(8), str(5)
memory usage: 11.8 MB
None

Missing Values:
PatientId         0
AppointmentID     0
Gender            0
Scheduled

Fix Column Names

In [5]:
# Rename columns for consistency
df.rename(columns={
    'PatientId': 'patient_id',
    'AppointmentID': 'appointment_id',
    'Gender': 'gender',
    'ScheduledDay': 'scheduled_day',
    'AppointmentDay': 'appointment_day',
    'Age': 'age',
    'Neighbourhood': 'neighbourhood',
    'Scholarship': 'scholarship',
    'Hipertension': 'hypertension',
    'Diabetes': 'diabetes',
    'Alcoholism': 'alcoholism',
    'Handcap': 'handicap',
    'SMS_received': 'sms_received',
    'No-show': 'no_show'
}, inplace=True)

print("Columns renamed ✅")
print(df.columns.tolist())

Columns renamed ✅
['patient_id', 'appointment_id', 'gender', 'scheduled_day', 'appointment_day', 'age', 'neighbourhood', 'scholarship', 'hypertension', 'diabetes', 'alcoholism', 'handicap', 'sms_received', 'no_show']


 Fix Data Types

In [6]:
# Convert dates to datetime
df['scheduled_day'] = pd.to_datetime(df['scheduled_day'])
df['appointment_day'] = pd.to_datetime(df['appointment_day'])

# Convert target to binary: No=0 (showed up), Yes=1 (no show)
df['no_show_flag'] = (df['no_show'] == 'Yes').astype(int)

print("Data types fixed ✅")
print(df[['scheduled_day','appointment_day','no_show_flag']].head())

Data types fixed ✅
              scheduled_day           appointment_day  no_show_flag
0 2016-04-29 18:38:08+00:00 2016-04-29 00:00:00+00:00             0
1 2016-04-29 16:08:27+00:00 2016-04-29 00:00:00+00:00             0
2 2016-04-29 16:19:04+00:00 2016-04-29 00:00:00+00:00             0
3 2016-04-29 17:29:31+00:00 2016-04-29 00:00:00+00:00             0
4 2016-04-29 16:07:23+00:00 2016-04-29 00:00:00+00:00             0


Fix Age Outliers

In [7]:
print("Age distribution before cleaning:")
print(df['age'].describe())

# Remove invalid ages
df = df[df['age'] >= 0]
df = df[df['age'] <= 100]

print("\nAge distribution after cleaning:")
print(df['age'].describe())
print("Shape after cleaning:", df.shape)

Age distribution before cleaning:
count    110527.000000
mean         37.088874
std          23.110205
min          -1.000000
25%          18.000000
50%          37.000000
75%          55.000000
max         115.000000
Name: age, dtype: float64

Age distribution after cleaning:
count    110519.000000
mean         37.084519
std          23.103165
min           0.000000
25%          18.000000
50%          37.000000
75%          55.000000
max         100.000000
Name: age, dtype: float64
Shape after cleaning: (110519, 15)


 Feature Engineering

In [8]:
# 1. Waiting days between scheduling and appointment
df['waiting_days'] = (df['appointment_day'] - df['scheduled_day']).dt.days

# Remove negative waiting days (data errors)
df = df[df['waiting_days'] >= 0]

# 2. Age group
df['age_group'] = pd.cut(df['age'],
    bins=[0, 12, 18, 35, 55, 100],
    labels=['Child', 'Teen', 'Young Adult', 'Middle Age', 'Senior'])

# 3. Appointment day of week
df['appointment_weekday'] = df['appointment_day'].dt.day_name()

# 4. Scheduled month
df['scheduled_month'] = df['scheduled_day'].dt.month_name()

# 5. Is same day appointment
df['same_day'] = (df['waiting_days'] == 0).astype(int)

# 6. Waiting day bucket
df['waiting_bucket'] = pd.cut(df['waiting_days'],
    bins=[-1, 0, 7, 30, 365],
    labels=['Same Day', '1-7 Days', '8-30 Days', '30+ Days'])

# 7. Chronic condition flag
df['has_chronic'] = ((df['hypertension'] == 1) |
                     (df['diabetes'] == 1)).astype(int)

# 8. Patient visit count (returning patients)
visit_counts = df.groupby('patient_id')['appointment_id'].count().reset_index()
visit_counts.columns = ['patient_id', 'total_visits']
df = df.merge(visit_counts, on='patient_id', how='left')

# 9. Returning patient flag
df['is_returning'] = (df['total_visits'] > 1).astype(int)

print("Feature engineering complete ✅")
print("New shape:", df.shape)
print("\nNew features:", ['waiting_days','age_group','appointment_weekday',
                          'scheduled_month','same_day','waiting_bucket',
                          'has_chronic','total_visits','is_returning'])

Feature engineering complete ✅
New shape: (71954, 24)

New features: ['waiting_days', 'age_group', 'appointment_weekday', 'scheduled_month', 'same_day', 'waiting_bucket', 'has_chronic', 'total_visits', 'is_returning']


 Save Cleaned Data

In [9]:
df.to_csv('cleaned_noshow_data.csv', index=False)
print("Cleaned data saved ✅")
print("Final shape:", df.shape)
print("\nNo-show rate:", df['no_show_flag'].mean()*100, "%")

Cleaned data saved ✅
Final shape: (71954, 24)

No-show rate: 28.51683019707035 %
